# F1-scientific-python — Practice p03 — Solution

In [ ]:
import numpy as np

SEED = 20260804
rng = np.random.default_rng(SEED)

In [ ]:
# Given: loop version 1 — a table of gaps between two sets of readings.
a = np.array([2.0, 9.0, 4.0, 7.0])
b = np.array([1.0, 3.0, 8.0])

gaps_loop = np.zeros((4, 3))
for i in range(4):
    for j in range(3):
        gaps_loop[i, j] = abs(a[i] - b[j])
print(gaps_loop)

In [ ]:
# Given: loop version 2 — shipping cost per parcel.
# Flat 5.00 up to 2 kg; heavier parcels add 2.00 per kg above 2 kg.
weights = np.array([0.5, 1.0, 2.0, 3.5, 6.0, 0.25])

costs_loop = []
for w in weights:
    if w <= 2.0:
        costs_loop.append(5.0)
    else:
        costs_loop.append(5.0 + 2.0 * (w - 2.0))
costs_loop = np.array(costs_loop)
print(costs_loop)

**Task A — kill the nested loop.** Recompute the `(4, 3)` gap table as
**`gaps`** in a single loop-free expression. Hint: give `a` a size-1 column
axis with `[:, None]` so it broadcasts against `b`. Then set **`same_a`** to
`np.array_equal(gaps_loop, gaps)`.

In [ ]:
gaps = np.abs(a[:, None] - b[None, :])
same_a = np.array_equal(gaps_loop, gaps)
print(gaps)
print("agrees with loop:", same_a)

`a[:, None]` is `(4, 1)` and `b[None, :]` is `(1, 3)`; broadcasting stretches both to `(4, 3)`, pairing every `a[i]` with every `b[j]` exactly as the nested loop did. `np.abs` then applies elementwise.

**Task B — kill the if/else loop.** Recompute the shipping costs as
**`costs`** with one `np.where` expression, and set **`same_b`** to
`np.array_equal(costs_loop, costs)`.

In [ ]:
costs = np.where(weights <= 2.0, 5.0, 5.0 + 2.0 * (weights - 2.0))
same_b = np.array_equal(costs_loop, costs)
print(costs)
print("agrees with loop:", same_b)

`np.where` evaluates both formulas for every parcel and picks, elementwise, the flat rate where the mask is True and the heavy-parcel formula where it is False — the loop's if/else without the loop.

**Task C — loop-free counting.** Without any loop, set:

- **`close_count`** — how many entries of `gaps` are smaller than 2
- **`cheap_fraction`** — the fraction of parcels costing at most 6.00

In [ ]:
close_count = (gaps < 2).sum()
cheap_fraction = (costs <= 6.0).mean()
print(close_count, cheap_fraction)

A comparison gives a mask; `.sum()` counts its True entries and `.mean()` gives the fraction that are True — the two standard loop-free counting idioms.

### Answer check

In [ ]:
assert gaps.shape == (4, 3) and same_a is True
assert np.array_equal(gaps, np.abs(a[:, None] - b[None, :]))
assert same_b is True and costs.shape == weights.shape
assert np.isclose(costs[-1], 5.0)          # 0.25 kg parcel: flat rate
assert np.isclose(costs[4], 5.0 + 2.0 * 4.0)
assert close_count == (np.abs(a[:, None] - b) < 2).sum()
assert np.isclose(cheap_fraction, (costs_loop <= 6.0).mean())
print("all checks passed")